# Re-score the keep_c2 ensemble — **no training**

This is a forward pass over the test split. A few minutes, CPU is fine.

## Why it is needed

The `scorecard.json` in your `results_keep_c2` folder was written **before**
calibration — QPSK reads 0.477 recall / 0.792 precision there, which is a high
threshold, against 0.847 / 0.393 at the calibrated ones. Publishing it would put
pre-calibration numbers next to post-calibration thresholds on the web page.

It can't be regenerated on the work machine either: that `data/processed` is the
Sep-22 RadChar-fix build, and it differs from `eavan-retrain` on exactly the radar
windows. Measured — the civilian recalls match to four decimals (BPSK 0.8291,
QPSK 0.8471, 16QAM 0.8500, 64QAM 0.8584) but LFM_RADAR reads 0.8082 instead of
0.8415.

So: same checkpoints, same thresholds, **correct data**, one run, both files.

## 1. Code

In [ ]:
BRANCH = 'main'

%cd /content
!rm -rf sedicAI_NEXA
!git clone -q -b $BRANCH https://github.com/eavan127/sedicAI_NEXA.git
%cd /content/sedicAI_NEXA
!git log --oneline -1

In [ ]:
!pip install -q pyyaml h5py

## 2. Data and checkpoints from Drive

`CKPT_SRC` is wherever `results_keep_c2` lives on your Drive. Adjust if the folder
name differs.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DATA = '/content/drive/MyDrive/sedic/eavan-retrain'
CKPT_SRC   = '/content/drive/MyDrive/sedic/results_keep_c2'

!mkdir -p data/processed results
!cp $DRIVE_DATA/X.npy data/processed/
!cp $DRIVE_DATA/y.npy data/processed/
!cp $DRIVE_DATA/snr_labels.npy data/processed/
!cp $CKPT_SRC/ensemble_*.pt results/
!cp $CKPT_SRC/best_model.pt results/
!ls -la results/ data/processed/

## 3. Set the architecture flag and the calibrated thresholds

A fresh clone carries whatever is committed, which is not necessarily what these
checkpoints were trained and calibrated with. Both have to match or the numbers are
meaningless — and with the wrong flag the checkpoints will not even load.

In [ ]:
!sed -i 's/^  stft_keep_rows: .*/  stft_keep_rows: true/'     configs/default.yaml
!sed -i 's/^  stft_freq_summary: .*/  stft_freq_summary: false/' configs/default.yaml
!sed -i 's/^  cumulant_features: .*/  cumulant_features: false/' configs/default.yaml

import re, pathlib
THRESHOLDS = {'BPSK': 0.16, 'QPSK': 0.16, '16QAM': 0.19, '64QAM': 0.2,
              'LFM_RADAR': 0.24, 'FHSS': 0.24, 'JAMMING': 0.89, 'NOISE_FLOOR': 0.16}
p = pathlib.Path('configs/default.yaml'); text = p.read_text()
for cls, v in THRESHOLDS.items():
    text, n = re.subn(rf'^(\s+){cls}: [0-9.]+$', rf'\g<1>{cls}: {v}', text, count=1, flags=re.M)
    assert n == 1, f'{cls} not replaced'
p.write_text(text)
print('thresholds set')
!grep -n 'stft_freq_summary:\|stft_keep_rows:\|cumulant_features:' configs/default.yaml

**Verify before scoring.** 181,898 parameters and a clean `strict=True` load is
what proves the config matches these checkpoints.

In [ ]:
import torch, json
from src.config import CFG, CLASSES
from src.models.amc_cnn import AMC_CNN

m = AMC_CNN(num_classes=len(CLASSES), input_len=CFG['signal']['window_len'])
n = sum(p.numel() for p in m.parameters())
print('parameters:', f'{n:,}', '  fc1.in:', m.fc1.in_features)
m.load_state_dict(torch.load('results/ensemble_0.pt', map_location='cpu'))
print('ensemble_0.pt loads strict=True: OK')
assert n == 181898, 'architecture does not match the keep_c2 checkpoints'

want = THRESHOLDS
got = CFG['multilabel_thresholds_per_class']
assert all(abs(got[k] - v) < 1e-9 for k, v in want.items()), got
print('thresholds match:', got)
del m

## 4. Score

Writes `evals/scorecard.json` and `evals/ensemble_scorecard.json` — both from this
one run, so their provenance agrees.

In [ ]:
!python -m src.evaluate --ensemble --n-models 5

Expect LFM_RADAR near **0.8415**, FHSS near **0.8291**, JAMMING near
**0.8399** — the `eval_rows_ens.json` figures. If LFM_RADAR comes out near 0.8082
instead, `data/processed` is the wrong build and `DRIVE_DATA` needs fixing.

In [ ]:
import json, pathlib
sc = json.loads(pathlib.Path('evals/scorecard.json').read_text())
b = sc['benchmark']
j = b.get('judged', b.get('judged_classes', {}))
print(f"{'class':<12} {'recall':>8}   expected")
print('-' * 36)
exp = {'LFM_RADAR': 0.8415, 'FHSS': 0.8291, 'JAMMING': 0.8399}
for c, v in j.items():
    r = v['recall'] if isinstance(v, dict) else v
    print(f'{c:<12} {r:>8.4f}   ~{exp.get(c, 0):.4f}')
print('\noverall:', 'PASSED' if b.get('passed') else 'FAILED')

## 5. Send them back

Two small JSON files. Put them in your local `evals/` folder, then rebuild the web
page (`python web/build.py`) so the published numbers come from the same data the
model was calibrated on.

In [ ]:
OUT = '/content/drive/MyDrive/sedic/evals_rows_c2'
!mkdir -p $OUT
!cp evals/scorecard.json evals/ensemble_scorecard.json $OUT/
!cp evals/accuracy_vs_snr.png evals/confusion_matrix.png $OUT/ 2>/dev/null
!ls -la $OUT/

In [ ]:
# Or download straight to your machine instead of Drive.
# from google.colab import files
# files.download('evals/scorecard.json')
# files.download('evals/ensemble_scorecard.json')